In [3]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent  # notebooks -> project root
# sys.path.insert(0, str(project_root))

# from clinical_synopsis.embedder import Embedder
# print("embedder import OK")

# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


# 0. Data in RETRIEVAL folder

Each file in `data/retrieval` supports one part of your RAG retrieval stack. 
- `minsearch_index.pkl` + `minsearch_documents.json` power **lexical search**.  
- `vector_index.npz` + `vector_index_metadata.json` power **semantic search**.  
- `metadata.db` is extra bookkeeping metadata about what you indexed.

Specifically:

- `minsearch_index.pkl` – the **lexical search index**  
  - A pickled Python object created by `minsearch.Index.fit(...)`.  
  - It contains the TF‑IDF / BM25–style structures that power your `search()` function.  
  - When you call `index = load_index()`, this is the file being loaded.

- `vector_index.npz` – the **embedding matrix and chunk IDs**  
  - A NumPy `.npz` archive with arrays like:
    - `embeddings`: a big 2D array, one embedding vector per chunk.
    - `chunk_ids`: the list of chunk IDs, aligned row‑by‑row with `embeddings`.  
  - When you do `vector_embeddings, vector_documents = load_vector_index()`, this file provides `vector_embeddings`.

- `vector_index_metadata.json` – the **metadata for each embedding row**  
  - JSON with a `documents` list. Each item is a dict for one chunk, with fields like:
    - `chunk_id`
    - `patient_id`
    - `doc_type`
    - `title`, `heading`
    - `is_oncology`
    - `chunk_text`, etc.  
  - `load_vector_index()` uses `chunk_ids` from `.npz` to reorder this list so `vector_documents[i]` matches `vector_embeddings[i]`.

- `minsearch_documents.json` – the **documents used to build the lexical index**  
  - JSON list of the chunk dictionaries that were fed into `minsearch.Index.fit(...)`.  
  - It’s basically the corpus for your lexical index, saved so you can inspect or rebuild it later.

- `metadata.db` – a **small database of higher-level document/patient metadata**  
  - Likely a SQLite DB (given the `.db` name) that stores additional information:
    - which files/patients were processed,
    - maybe source paths, dates, or other indexing metadata.  
  - It’s not used directly in `rag.py`, but is useful for bookkeeping and possibly other scripts in your project.




In [4]:
!ls /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval

metadata.db		  minsearch_index.pkl  vector_index_metadata.json
minsearch_documents.json  vector_index.npz


## What to included on GitHub

What should go into `data/` on GitHub depends on whether you want the repo to be:

- **fully runnable immediately**, or
- **lighter-weight but rebuildable from code**.

For a RAG project, reproducibility usually means including either the derived retrieval artifacts or clear code and instructions to regenerate them. [github](https://github.com/gchure/reproducible_research)

## If you want it to run right away

Then yes, you should include the retrieval artifacts your `rag.py` directly depends on:

- `data/retrieval/minsearch_index.pkl`
- `data/retrieval/vector_index.npz`
- `data/retrieval/vector_index_metadata.json`

And likely also:

- `data/retrieval/minsearch_documents.json`
- `data/retrieval/metadata.db`

because they help inspection, rebuilding, or explainability.

If those files are present, someone cloning the repo can run `rag.py` without rebuilding the indexes first.

## Minimum needed by `rag.py`

Strictly speaking, your current `rag.py` only directly loads:

- `minsearch_index.pkl`
- `vector_index.npz`
- `vector_index_metadata.json`

So those three are the true runtime minimum for the current script.

## But should that be all in `data/`?

Probably not the entire `data/` folder. A cleaner GitHub setup is usually:

- keep only the **small, necessary, non-sensitive derived artifacts** that make the project runnable,
- exclude big temporary or rebuildable files,
- and explain in `README.md` how to regenerate indexes if needed. Good reproducible-research structure usually separates raw, derived, and scratch data and documents what each artifact is for. [audreyrpark.github](https://audreyrpark.github.io/RPl-Spielman-2020/template_readme.html)

## Practical recommendation for your project

For your course project, I would suggest:

### Include
- `data/retrieval/minsearch_index.pkl`
- `data/retrieval/vector_index.npz`
- `data/retrieval/vector_index_metadata.json`
- maybe `data/retrieval/minsearch_documents.json` if it is not too large

### Optional
- `metadata.db`, only if another script actually uses it or it helps explain the pipeline

### Also include
- the preprocessing/index-building scripts
- a short README note saying these are derived retrieval artifacts generated from your processed patient documents

## One thing to check first

Before pushing, check file sizes. GitHub is fine with normal files, but large binary artifacts can become awkward to version. If `vector_index.npz` is large, you may prefer to:
- leave it out,
- and provide a script to regenerate it instead.

So the short answer is:

- **No, not automatically “all that and only that.”**
- **Yes, those three retrieval files are the minimum needed for your current `rag.py` to run.**

A good next step is to list the sizes of those files and decide whether you want a “runs immediately” repo or a “rebuild first” repo.

# 1. Inspect retrieval of chunks with exact patient IDs

To make notebook testing easier, create a tiny helper cell to copy one exact patient ID from there each time to avoid patient id mismatch.
```
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
```
(**Where's vector_documents from?**)

You’re now at the point where it makes sense to compare:

- lexical,

- semantic,

- hybrid

for the same patient and same query. That is the right setup for the notebook experiment before you build the CSV evaluation runner.


`rag` is the module you imported with `import rag as rag`. It contains:
- vector_embeddings
- vector_documents
and the `rag()` function itself.

`rag.vector_documents` was created in the module by:
```python
vector_embeddings, vector_documents = load_vector_index()
```
`vector_documents` is a Python list of dictionaries, one per chunk, loaded from `vector_index_metadata.json` (in `data/retrieval/vector_index_metadata.json`). Each dict has metadata like:
- chunk_id
- patient_id
- doc_type
- title
- heading
- is_oncology
etc.

For testing
`rag.vector_documents[0]`
takes the first chunk in that list. That’s just an arbitrary but valid example chunk.

`rag.vector_documents[0]["patient_id"]`
reads the "patient_id" field from that first chunk’s metadata. This gives you a concrete patient ID string that you know exists in both:
- the lexical index (minsearch), and
- the vector index metadata.


So `vector_documents` is “the metadata that corresponds row‑by‑row to vector_embeddings, built using both the .npz file and its metadata JSON,” not as being loaded from the .npz alone.

In the folder `data/retrieval`:
- `vector_index.npz` holds the embedding matrix and the chunk ID list.
- `vector_index_metadata.json` holds the documents list with metadata, including patient_id.
Your `load_vector_index()` reads both and then wires them together: 
```python
def load_vector_index():
    data = np.load(VECTOR_INDEX_PATH, allow_pickle=True)
    with open(VECTOR_METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    embeddings = data["embeddings"]
    chunk_ids = data["chunk_ids"].tolist()
    documents = metadata["documents"]

    docs_by_chunk_id = {doc["chunk_id"]: doc for doc in documents}
    ordered_docs = [docs_by_chunk_id[chunk_id] for chunk_id in chunk_ids]

    return embeddings, ordered_docs
```
So:
- `vector_embeddings` comes from `vector_index.npz` (embeddings array).
- `vector_documents` comes from `vector_index_metadata.json` (`metadata["documents"]`), but reordered to match the embedding rows according to chunk_ids.
Conceptually, for your notebook:
- `vector_embeddings[i]` = embedding vector for chunk i, stored in vector_index.npz,
- `vector_documents[i]` = dict with patient_id, doc_type, etc. for the same chunk, coming from the JSON.

That's why pid is a real patient ID attached to the same chunk as `vector_embeddings[0]`.

In [5]:
import rag as rag

# Look at the first chunk we have in the vector index, take the patient_id attached to that chunk:
pid = rag.vector_documents[0]["patient_id"]

result = rag.rag(
    query="What oncology-related events are documented?",
    patient_id=pid,
    search_type="hybrid",
    num_results=5,
)

len(result["search_results"]), result["search_results"][:2]

(5,
 [{'id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'chunk_id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': 'eacfd84f2024e811caae390e056a4e52fefc5249',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Aurora248 Dooley940',
   'heading': 'Oncology Timeline: Aurora248 Dooley940',
   'chunk_text': '- Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082},
  {'chunk_id': 'f50d5c213d71cb54157be1dbe78d05ff2cb23489',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': '304ddea3daed73b3c0c0f47535fef1906ea22987',
   'doc_type': 'oncology_timeline_events',
   'title': 'oncology_timeline_events.csv',
   'heading': 'oncology_timeline_events',
   'chunk_text': "event_type: Observation; date: 2015-10-01T05:29:59-04:00; label:

In [6]:
print("Answer cost (USD):", result["answer_total_cost_usd"])
print("Eval cost (USD):  ", result["eval_total_cost_usd"])
print("Overall cost (USD):", result["overall_total_cost_usd"])

Answer cost (USD): 0.0028627500000000003
Eval cost (USD):   0.00220575
Overall cost (USD): 0.0050685


In [7]:
print("Answer tokens (in/out/total):",
      result["prompt_tokens"],
      result["completion_tokens"],
      result["total_tokens"])

print("Eval tokens (in/out/total):",
      result["eval_prompt_tokens"],
      result["eval_completion_tokens"],
      result["eval_total_tokens"])

Answer tokens (in/out/total): 1951 311 2262
Eval tokens (in/out/total): 2389 92 2481


In [8]:
for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Chunk ID:", doc.get("chunk_id"))
    print("Patient ID:", doc.get("patient_id"))
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Text:", doc.get("chunk_text", "")[:500])

Rank: 1
Chunk ID: 6e3f2f8c188a50ec8d84a45b499a14a631fcc414
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Dooley940
Heading: Oncology Timeline: Aurora248 Dooley940
Text: - Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
- Oncology-related dated events: 52
Rank: 2
Chunk ID: f50d5c213d71cb54157be1dbe78d05ff2cb23489
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline_events
Title: oncology_timeline_events.csv
Heading: oncology_timeline_events
Text: event_type: Observation; date: 2015-10-01T05:29:59-04:00; label: Cancer Disease Progression; status: Patient's condition improved; resource_id: 3ee3c67c-a4a3-37e1-e49b-f20f32cc14db; source_file: data/prototype/sample50/Aurora248_Dooley940_03b93198-d95e-c385-c3a7-80470f411d18.json
Rank: 3
Chunk ID: 561c5809a7e9251705306dee52b08e9fd64721ce
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Doole

In [9]:
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
available_patient_ids[:10]

['03b93198-d95e-c385-c3a7-80470f411d18',
 '0c0f2095-e8ab-7ac4-6ef4-625748255480',
 '0f5704ee-b38b-5a68-449d-9c44806517d0',
 '188e1f01-15b7-d51b-c76d-bdd7772a10e9',
 '25197dc8-9425-1999-5914-f2171b0d4e32',
 '263375ec-5856-81b8-9e51-1cb8e8bcba30',
 '29f6beee-162f-0113-7884-72245814693f',
 '397b2de6-ccd8-858f-bf4a-b6fc379589bd',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 '3af995f1-02a5-07ee-5a7e-e2470a017f1e']

In [10]:
# # running the same query for just the first patient id in the list of available patient ids
# # which is what we did above with pid

# result = rag.rag(
#     query="What oncology-related events are documented?",
#     patient_id=available_patient_ids[0],
#     is_oncology=True,
#     search_type="hybrid",
#     num_results=5,
# )

# for i, doc in enumerate(result["search_results"], start=1):
#     print("=" * 80)
#     print("Rank:", i)
#     print("Doc type:", doc.get("doc_type"))
#     print("Title:", doc.get("title"))
#     print("Heading:", doc.get("heading"))
#     print("Chunk ID:", doc.get("chunk_id"))
#     print("Text:", doc.get("chunk_text", "")[:800])

Your test_cases now need to use real patient IDs from the index, not placeholder IDs. For example:

(Note for the below selection:
Rows for the second and third questions (is_oncology=None) can include any chunks, including ones where the is_oncology field is missing in your underlying data.)

In [11]:
# 3 test cases for the first patient id in the list of available patient ids
test_cases = [
    {
        "patient_id": available_patient_ids[0],
        "query": "What oncology-related events are documented?",
        "is_oncology": True,
    },
    {
        "patient_id": available_patient_ids[0],
        "query": "What recent conditions are documented?",
        "is_oncology": None, # so retrieval may pick chunks where is_oncology is 0, 1, or entirely absent.
    },
    {
        "patient_id": available_patient_ids[0],
        "query": "What medications are mentioned?",
        "is_oncology": None,
    },
]

A df or more debugging, so you can see:

- whether retrieval returned anything,

- what kind of document came first,

- and whether the answer quality changes across search types.

In [12]:
import pandas as pd

# this took 38s

rows = []

for case in test_cases:
    for search_type in ["lexical", "semantic", "hybrid"]:
        result = rag.rag(
            query=case["query"],
            patient_id=case["patient_id"],
            is_oncology=case["is_oncology"],
            search_type=search_type,
            num_results=5,
        )

        rows.append({
            "patient_id": case["patient_id"],
            "query": case["query"],
            "is_oncology": case["is_oncology"],
            "search_type": search_type,

            "answer": result["answer"],
            "relevance": result.get("relevance"),
            "groundedness": result.get("groundedness"),
            "response_time": result.get("response_time"),

            "n_search_results": len(result.get("search_results", [])),
            "top_doc_type": result["search_results"][0].get("doc_type")
                if result.get("search_results") else None,
            "top_title": result["search_results"][0].get("title")
                if result.get("search_results") else None,

            # Answer tokens and cost
            "answer_prompt_tokens": result.get("prompt_tokens"),
            "answer_completion_tokens": result.get("completion_tokens"),
            "answer_total_tokens": result.get("total_tokens"),
            "answer_input_cost_usd": result.get("answer_input_cost_usd"),
            "answer_output_cost_usd": result.get("answer_output_cost_usd"),
            "answer_total_cost_usd": result.get("answer_total_cost_usd"),

            # Eval tokens and cost
            "eval_prompt_tokens": result.get("eval_prompt_tokens"),
            "eval_completion_tokens": result.get("eval_completion_tokens"),
            "eval_total_tokens": result.get("eval_total_tokens"),
            "eval_input_cost_usd": result.get("eval_input_cost_usd"),
            "eval_output_cost_usd": result.get("eval_output_cost_usd"),
            "eval_total_cost_usd": result.get("eval_total_cost_usd"),

            # Combined cost
            "overall_total_cost_usd": result.get("overall_total_cost_usd"),
        })

df = pd.DataFrame(rows)
df

,patient_id,query,is_oncology,search_type,answer,relevance,groundedness,response_time,n_search_results,top_doc_type,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,03b93198-d95e-c385-c3a7-80470f411d18,What oncology-related events are documented?,True,lexical,The oncology timeline documents these events:\...,RELEVANT,GROUNDED,4.729512,5,oncology_timeline,...,0.002148,0.001152,0.003300,3247,73,3320,0.002435,0.000329,0.002764,0.006064
1,03b93198-d95e-c385-c3a7-80470f411d18,What oncology-related events are documented?,True,semantic,The oncology-related events documented in the ...,RELEVANT,GROUNDED,3.455765,5,oncology_timeline_events,...,0.000842,0.000869,0.001710,1442,63,1505,0.001081,0.000284,0.001365,0.003075
2,03b93198-d95e-c385-c3a7-80470f411d18,What oncology-related events are documented?,True,hybrid,The oncology-related events documented in the ...,RELEVANT,GROUNDED,3.800643,5,oncology_timeline,...,0.001463,0.001409,0.002872,2391,68,2459,0.001793,0.000306,0.002099,0.004971
3,03b93198-d95e-c385-c3a7-80470f411d18,What recent conditions are documented?,None,lexical,The recent conditions documented in the **cond...,RELEVANT,PARTLY_GROUNDED,3.985629,5,conditions,...,0.001234,0.000770,0.002003,1943,132,2075,0.001457,0.000594,0.002051,0.004054
4,03b93198-d95e-c385-c3a7-80470f411d18,What recent conditions are documented?,None,semantic,The recent conditions documented in the **Pati...,RELEVANT,GROUNDED,3.599205,5,oncology_timeline,...,0.001064,0.001053,0.002117,1780,59,1839,0.001335,0.000266,0.001600,0.003718
5,03b93198-d95e-c385-c3a7-80470f411d18,What recent conditions are documented?,None,hybrid,Recent conditions documented in the **Patient ...,RELEVANT,GROUNDED,3.911117,5,conditions,...,0.001147,0.001206,0.002353,1924,75,1999,0.001443,0.000337,0.001780,0.004133
6,03b93198-d95e-c385-c3a7-80470f411d18,What medications are mentioned?,None,lexical,The medications mentioned in the **medications...,RELEVANT,GROUNDED,3.221606,5,medications,...,0.001274,0.000599,0.001873,1960,66,2026,0.001470,0.000297,0.001767,0.003640
7,03b93198-d95e-c385-c3a7-80470f411d18,What medications are mentioned?,None,semantic,The medications mentioned in the record includ...,RELEVANT,GROUNDED,3.934580,5,patient_overview,...,0.001345,0.000878,0.002222,2115,74,2189,0.001586,0.000333,0.001919,0.004142
8,03b93198-d95e-c385-c3a7-80470f411d18,What medications are mentioned?,None,hybrid,The medications mentioned in the record are:\n...,RELEVANT,GROUNDED,3.486814,5,medications,...,0.001371,0.000963,0.002334,2169,62,2231,0.001627,0.000279,0.001906,0.004240


In [13]:
df.groupby("search_type")["overall_total_cost_usd"].sum()


search_type
hybrid      0.013344
lexical     0.013758
semantic    0.010934
Name: overall_total_cost_usd, dtype: float64

In [14]:
df.groupby("search_type")[["answer_total_cost_usd", "eval_total_cost_usd"]].mean()

,answer_total_cost_usd,eval_total_cost_usd
search_type,,
hybrid,0.002520,0.001928
lexical,0.002392,0.002194
semantic,0.002017,0.001628


# 2. Evaluation

You can apply exactly the same pattern from the course: define a small ground-truth table for retrieval, then compute hit rate@K and MRR@K over that table for each search mode (lexical, semantic, hybrid). The only difference is what you use as the “relevant” item for each question in your EHR setting. Hit rate@K and MRR@K are standard retrieval metrics for RAG, so your approach generalizes directly.

1. Define retrieval ground truth
In the course you had something like:

- A table where each row has: query, relevant_doc_id.

For your project, a row can be:

- patient_id
- question
- one or more gold chunk IDs or gold doc identifiers (the chunks that definitely contain the answer)

Example schema (as a Pandas-like table):

- patient_id: "PATIENT_001"
- question: "What oncology-related events are documented?"
- gold_chunk_ids: ["chunk-123", "chunk-456"]

You can store this as JSON or CSV, e.g.:

```json
[
  {
    "patient_id": "PATIENT_001",
    "question": "What oncology-related events are documented?",
    "gold_chunk_ids": ["chunk-123", "chunk-456"]
  },
  {
    "patient_id": "PATIENT_001",
    "question": "What medications are mentioned?",
    "gold_chunk_ids": ["chunk-789"]
  }
]
```

2. Run retrieval only, per search mode
For each ground-truth row, you want:
- the list of retrieved chunk IDs for each search mode,
- in rank order.

Use your existing search functions directly (without calling rag()):
(This gives you a ranked list of chunk_ids for each (patient_id, question, search_type).)
```python
def get_ranked_chunk_ids(query, patient_id, search_type, k=10):
    if search_type == "lexical":
        results = rag.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = rag.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = rag.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be lexical, semantic, or hybrid")

    return [doc["chunk_id"] for doc in results]
```

3. Compute Hit rate@K and MRR@K
Recall the definitions:
- **Hit rate@K**: for each query, 1 if any relevant chunk is in the top K, else 0; average over queries.
- **MRR@K**: for each query, find the rank of the first relevant chunk in the top K, use 
1/rank; 0 if no relevant chunk in top K; average over queries.

In code (for your notebook):
```python
def eval_hit_mrr_for_mode(gt_rows, search_type, k=10):
    """gt_rows is a list of dicts with keys:
       - patient_id
       - question
       - gold_chunk_ids (list of strings)
    """
    hits = []
    reciprocal_ranks = []

    for row in gt_rows:
        patient_id = row["patient_id"]
        question = row["question"]
        gold_ids = set(row["gold_chunk_ids"])

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        # Hit@K
        hit = int(any(doc_id in gold_ids for doc_id in retrieved_ids))
        hits.append(hit)

        # MRR@K
        rr = 0.0
        for rank, doc_id in enumerate(retrieved_ids, start=1):
            if doc_id in gold_ids:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

    hit_rate = sum(hits) / len(hits) if hits else 0.0
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0.0
    return hit_rate, mrr
```

Then evaluate all three modes:
```python
search_types = ["lexical", "semantic", "hybrid"]

for st in search_types:
    hit_k, mrr_k = eval_hit_mrr_for_mode(gt_rows, st, k=5)
    print(f"{st}: Hit@5={hit_k:.3f}, MRR@5={mrr_k:.3f}")
```

This replicates the course pattern: for each “ground-truth record” (your patient–question–gold evidence row), you get a retrieval relevance score per search mode.

4. Use the metrics for tuning
Once you have Hit@K and MRR@K per search mode, you can use them exactly as in the course to tune search parameters, for example:
- in `search()`: adjust `boost_dict` weights (`title`, `heading`, `chunk_text`),
- in `semantic_search()`: adjust normalization or top‑N cutoff,
- in `hybrid_search()`: adjust `rrf_k`, adjust the number of lexical vs semantic candidates you pass into RRF.

The evaluation workflow is:

1) Define or update the ground-truth rows.

2) Run lexical / semantic / hybrid retrieval on those rows.

3) Compute Hit@K and MRR@K.

4) Tweak parameters and re-measure.

5) Pick the setting that gives the best metrics for your chosen K (e.g. K=5).


You’re right that in the course the “ground truth” basically covers the whole tiny toy dataset (72 docs × 5 questions = 360 query–label pairs). That’s just a convenient teaching setup, not a rule you must follow. In your project you usually only label a small, representative subset of your full EHR dataset.

What “ground truth” is for here
For your RAG/search evaluation, the ground truth is just:
- a set of queries (your questions), and
- for each query, the documents/chunks you mark as relevant.
Its job is to estimate how good your retrieval is, not to cover every single record the app will ever see.

**How big should your ground truth be?**
There is no fixed number. Typical guidance for ranking/RAG evaluation is:
- Start with something like 50–100 queries that:
    - cover the key use cases,
    - vary in difficulty and phrasing,
    - touch different types of documents/sections.
- If metrics are very noisy (change a lot when you tweak parameters), expand to more queries until performance curves stabilize.

For your project, a very sensible starting point is something like:
- pick, say, 5–10 patients you know well,
- for each, define 3–5 questions that reflect your app’s goals (e.g., “oncology history”, “medications”, “smoking status”),
- for each question, label one or a small set of gold chunks.
That already gives you 15–50 query–label pairs, enough to compare lexical vs semantic vs hybrid and do parameter tuning.

You already have a defined “population” of 50 patients; now you just need to choose a small subset of them (5–10) to label and evaluate on. That subset should be either random or deliberately representative.

1. What you’re sampling from
data/processed/mcode_breast_sample_50_manifest.csv is your list of 50 patients and their files. Conceptually:
- each row ≈ one patient,
- with columns like patient_id, file paths, etc.
Your ground-truth patients will be a subset of these rows; the full retrieval index can still include all 50.

2. Option A: simple random sample of 5–10 patients
If you don’t need special coverage (e.g., early vs late stage, different note types), the simplest is:
- Load the manifest CSV (e.g., in a notebook).
- Randomly sample, say, 8 patients.
- Use those patient IDs when you build your question–gold-chunk ground truth.
This is statistically clean and easy to explain in your report:
“From the 50 patients, we randomly selected 8 for manual ground-truth labeling and retrieval evaluation.”
```python
import pandas as pd

manifest_path = "data/processed/mcode_breast_sample_50_manifest.csv"
df = pd.read_csv(manifest_path)

# Sample 8 patients without replacement
sampled = df.sample(n=8, random_state=42)

sampled_patient_ids = sampled["patient_id"].tolist()
print(sampled_patient_ids)
```

3. Option B: deliberately pick “interesting” patients
Because your dataset is small, you might prefer to choose patients that:
- have oncology-related notes vs non-oncology notes,
- vary in number of documents or events,
- contain the kinds of information you care about (e.g., medications, treatments, staging).

A pragmatic recipe:
    1. Load the manifest and maybe join it with a summary of per-patient stats (e.g., count of chunks, presence of oncology flag).
    2. Manually inspect a few rows (or your CSV summaries) and pick:
        - 2–3 patients with many notes,
        - 2–3 with few notes,
        - 1–2 edge cases (e.g., unusual treatments).

You can document this as:
“We selected 7 patients to cover a range of note volumes and oncology characteristics.”

The trade-off:
- random sampling is cleaner for “unbiased” evaluation,
- hand-picking can ensure you actually exercise the key behaviors your app should support.

4. How this connects to ground truth size
Once you have your 5–10 patient IDs:
- For each selected patient, create 3–5 realistic questions.
- For each question, label the gold chunk_id(s).
That will give you somewhere around 15–50 query–label pairs.

You still index all 50 patients for retrieval, but you only evaluate metrics on that labeled subset. That’s exactly the intended pattern: a big searchable corpus, small labeled ground truth.

For option B, the best first step is a notebook cell that helps you inspect the manifest, summarize it per patient, and surface candidate patients to choose from. Since your goal is to pick “interesting” patients rather than sample randomly, you want code that shows things like number of files/rows per patient, note types, and any obviously useful columns.

In [15]:
# #we see that for every patient there is just 1 row

# import pandas as pd
# from IPython.display import display

# manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# # Load manifest
# df = pd.read_csv(manifest_path)

# print("Shape:", df.shape)
# print("\nColumns:")
# print(df.columns.tolist())

# print("\nFirst 5 rows:")
# display(df.head())

# print("\nMissing values per column:")
# display(df.isna().sum().sort_values(ascending=False))

# # Try to infer the patient column
# candidate_patient_cols = [c for c in df.columns if "patient" in c.lower() or c.lower().endswith("_id")]
# print("\nPossible patient ID columns:", candidate_patient_cols)

# # Pick the first likely patient ID column; change manually if needed
# if not candidate_patient_cols:
#     raise ValueError("Couldn't infer a patient ID column. Please inspect df.columns and set patient_col manually.")

# patient_col = candidate_patient_cols[0]
# print("\nUsing patient column:", patient_col)

# # Basic per-patient summary
# patient_summary = (
#     df.groupby(patient_col)
#       .agg(
#           n_rows=(patient_col, "size"),
#           n_unique_values=("source_file", "nunique") if "source_file" in df.columns else (patient_col, "size")
#       )
#       .sort_values("n_rows", ascending=False)
#       .reset_index()
# )

# # Add quick summaries for potentially useful columns
# useful_cols = [
#     c for c in df.columns
#     if c.lower() in {"doc_type", "document_type", "category", "section", "title", "source_file", "file_path"}
# ]

# for col in useful_cols:
#     if col != patient_col:
#         top_vals = (
#             df.groupby(patient_col)[col]
#               .apply(lambda s: ", ".join(map(str, s.dropna().astype(str).value_counts().head(3).index.tolist())))
#               .rename(f"top_{col}")
#         )
#         patient_summary = patient_summary.merge(top_vals, on=patient_col, how="left")

# print("\nPatients with the most rows/files:")
# display(patient_summary.head(15))

# print("\nPatients with the fewest rows/files:")
# display(patient_summary.tail(15))

# # Optional: inspect one patient in detail
# example_patient = patient_summary.iloc[0][patient_col]
# print(f"\nExample patient for closer inspection: {example_patient}")

# patient_view = df[df[patient_col] == example_patient].copy()
# display(patient_view.head(20))

# # Optional: if you want a mixed selection strategy, take:
# # - top 3 most complex patients
# # - bottom 2 simplest patients
# # - 3 from the middle
# n = len(patient_summary)
# mixed_candidates = pd.concat([
#     patient_summary.head(3),
#     patient_summary.iloc[max(n//2 - 1, 0): min(n//2 + 2, n)],
#     patient_summary.tail(2)
# ]).drop_duplicates(subset=[patient_col])

# print("\nSuggested mixed candidate set:")
# display(mixed_candidates)

# 9 patients for ground truth

We pick 9 patients with 3 from each complexity bucket (low, medium, high) for the ground-truth set, in order to cover simple, medium, and complex EHRs.

`n_resources` is the total number of FHIR resources in that patient’s bundle — i.e., how many individual clinical records (Patient, Encounter, Observation, Condition, Procedure, MedicationRequest, DiagnosticReport, etc.) are contained in the JSON file for that patient.

So we see that the higher `n_resources`, the higher `complexity_score`

Remember:
# Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.



In [16]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [17]:
df

,filename,patient_id,patient_name,n_resources,n_encounters,n_observations,n_conditions,n_procedures,n_medication_requests,n_medication_administrations,n_diagnostic_reports,first_date,last_date,followup_days,complexity_score,complexity_bucket,sample_seed
0,data/raw/longitudinalMCODEBreast/Corrie32_Boyl...,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,230,27,80,2,27,20,0,28,2020-12-18T03:22:18-05:00,2022-05-20T18:36:55-04:00,518,317,low,42
1,data/raw/longitudinalMCODEBreast/Florine959_St...,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,261,25,119,5,23,11,0,26,2019-06-11T21:25:37-04:00,2022-06-21T05:03:45-04:00,1105,330,low,42
2,data/raw/longitudinalMCODEBreast/Deana43_Baumb...,3f130449-d7db-f118-5bd0-cce81084e911,Deana43 Baumbach677,431,42,206,10,23,18,0,44,2012-12-18T03:09:54-05:00,2022-04-12T04:24:54-04:00,3402,540,low,42
3,data/raw/longitudinalMCODEBreast/Joni720_Stied...,43c173b0-172c-f414-5c62-1bdf4bb33954,Joni720 Stiedemann542,459,47,218,17,29,5,0,51,2014-04-22T07:35:47-04:00,2022-05-17T21:08:31-04:00,2947,579,low,42
4,data/raw/longitudinalMCODEBreast/Santos184_Jas...,64ae3769-65e4-222e-6793-1a3bc14ec682,Santos184 Jaskolski867,468,45,241,4,31,11,0,48,2010-08-02T09:36:48-04:00,2022-05-17T22:36:39-04:00,4306,584,low,42
5,data/raw/longitudinalMCODEBreast/Mónica985_Se...,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,436,61,151,6,54,8,0,62,2018-01-07T15:16:05-05:00,2022-04-21T02:52:37-04:00,1564,602,low,42
6,data/raw/longitudinalMCODEBreast/Maryellen651_...,af3bd539-de27-28d9-9016-f1643d4615c0,Maryellen651 Zboncak558,514,54,238,11,35,18,0,56,2011-08-29T07:05:19-04:00,2022-04-27T18:02:46-04:00,3894,660,low,42
7,data/raw/longitudinalMCODEBreast/Darcie474_Fra...,568ec0af-94fa-521b-012e-88f61f78028f,Darcie474 Frami345,496,69,180,2,59,11,0,71,2015-10-27T05:01:54-04:00,2022-04-25T17:02:50-04:00,2372,686,low,42
8,data/raw/longitudinalMCODEBreast/Jani266_Thiel...,3e693a9a-de55-de2e-aab9-500036bcf04b,Jani266 Thiel172,629,76,258,8,67,14,0,80,2011-05-02T16:40:40-04:00,2022-06-03T06:55:38-04:00,4049,844,low,42
9,data/raw/longitudinalMCODEBreast/Dolores502_Ca...,6fb374e8-33aa-a5ea-f050-b61394dfcb99,Dolores502 Caldera106,872,101,305,13,165,18,1,111,2006-05-18T17:21:05-04:00,2022-06-30T17:36:05-04:00,5887,1244,low,42


In [18]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


So we have a subset of 9 patients with a range of difficulty, which is exactly what you want when tuning retrieval parameters using Hit@K and MRR rather than evaluating only easy or only dense cases.



Pick a K that matches your prompt budget (e.g., if you typically feed 5 chunks into the LLM, use K=5)

In [20]:
# import math

# # Example: ground truth rows — fill these manually once you know chunk_ids
# gt_rows = [
#     # Replace with your real patient/question/gold_chunk_ids
#     {
#         "patient_id": "PATIENT_001",
#         "question": "Summarize the patient's oncology history.",
#         "gold_chunk_ids": ["chunk-id-1", "chunk-id-2"],
#     },
#     {
#         "patient_id": "PATIENT_002",
#         "question": "What medications is the patient taking?",
#         "gold_chunk_ids": ["chunk-id-10"],
#     },
#     # ...
# ]

# def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
#     """Run the chosen search and return top-k chunk_ids in rank order."""
#     if search_type == "lexical":
#         results = search(
#             query=query,
#             patient_id=patient_id,
#             num_results=k,
#         )
#     elif search_type == "semantic":
#         results = semantic_search(
#             query=query,
#             patient_id=patient_id,
#             num_results=k,
#         )
#     elif search_type == "hybrid":
#         results = hybrid_search(
#             query=query,
#             patient_id=patient_id,
#             num_results=k,
#         )
#     else:
#         raise ValueError("search_type must be one of: lexical, semantic, hybrid")

#     return [doc["chunk_id"] for doc in results]

# def hit_rate_at_k(retrieved_ids, relevant_ids, k):
#     """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
#     relevant = set(relevant_ids)
#     top_k = retrieved_ids[:k]
#     return int(any(doc_id in relevant for doc_id in top_k))

# def mrr_at_k(retrieved_ids, relevant_ids, k):
#     """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
#     relevant = set(relevant_ids)
#     for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
#         if doc_id in relevant:
#             return 1.0 / rank
#     return 0.0

# def eval_search_type(gt_rows, search_type, k=5):
#     hits = []
#     mrrs = []

#     for row in gt_rows:
#         patient_id = row["patient_id"]
#         question = row["question"]
#         gold_ids = row["gold_chunk_ids"]

#         retrieved_ids = get_ranked_chunk_ids(
#             query=question,
#             patient_id=patient_id,
#             search_type=search_type,
#             k=k,
#         )

#         hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
#         mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

#     hit_rate = sum(hits) / len(hits) if hits else math.nan
#     mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

#     return hit_rate, mrr

# # Evaluate lexical, semantic, and hybrid for K=5
# for st in ["lexical", "semantic", "hybrid"]:
#     hr, mrr = eval_search_type(gt_rows, st, k=5)
#     print(f"{st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

## Finding gold chunks

You choose `gold_chunk_ids` by **looking at your chunks for each patient and question and explicitly marking the ones that contain the evidence you want the retriever to find**, and you test enough questions to cover your main use cases without making labeling intractable — typically on the order of a few dozen, not hundreds. [linkedin](https://www.linkedin.com/posts/dan-bucureanu_here-are-the-easy-steps-to-perform-your-rag-activity-7341344334752985089-537X)

Here’s a practical way to do it for your 9 patients.

***

## 1. How to decide on gold_chunk_ids

For each selected patient and question:

1. **Pick a realistic clinical question**  
   Examples:
   - “Summarize the patient’s oncology history.”
   - “What treatments has the patient received?”
   - “What medications is the patient currently taking?”
   - “Does the patient have documented smoking status?”

2. **Use your retrieval or direct DB/CSV inspection to find candidate chunks**  
   - Run `search()` or `semantic_search()` for that patient and question and inspect the top K chunks.  
   - Or query `metadata.db` / look at the derived `.md`/`.csv` files directly.

3. **Mark the chunks that truly contain needed answer information**  
   For example:
   - For “oncology history,” the chunk(s) that list cancer diagnoses, staging, key events.
   - For “medications,” the chunk(s) that list drugs, doses, start dates.

4. **Add all such chunks to `gold_chunk_ids`**  
   - If one chunk has almost everything, you might have a single ID.
   - If the answer is spread across multiple sections (e.g., one chunk for diagnosis, one for treatment), include both. [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

Guidelines:

- Think of them as **“gold nugget” chunks**: if the retriever gets these, the LLM has enough factual material to answer correctly. [linkedin](https://www.linkedin.com/posts/dan-bucureanu_here-are-the-easy-steps-to-perform-your-rag-activity-7341344334752985089-537X)
- Don’t list *every* chunk that mentions the topic; focus on those that are clearly useful and needed.
- It’s fine if some questions have 1 gold chunk and others have 2–3; Hit@K and MRR work with sets of relevant items. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

***

## 2. How many questions to test

There’s no hard rule, but common practice for a small project like yours:

- **Per patient**: 3–5 questions that reflect your app’s intended use:
  - 1–2 on oncology history/events.
  - 1 on treatments/procedures.
  - 1 on medications.
  - 1 on another key aspect you care about (e.g., encounters timeline, labs). [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

- For **9 patients**:
  - If you do 3 questions each → ~27 questions.
  - If you do 4 each → ~36 questions.

That’s usually enough to:

- get reasonably stable Hit@K and MRR estimates,
- compare lexical vs semantic vs hybrid search,
- and tune a few parameters (boosts, `rrf_k`, K) without your labeling workload exploding. [dataaihub](https://www.dataaihub.co/learn/retrieval-evaluation)

Try to ensure your question set:

- covers different document types (`patient_overview`, `oncology_timeline`, `conditions`, `medications`, etc.),
- includes both easy and harder questions (some with answers in one chunk, some needing multiple chunks),
- includes both oncology and non-oncology questions so the `is_oncology` metadata actually matters. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

***

So a good target for your project is:

- 9 patients × 3–4 questions each → about 30 labeled question–gold-chunk sets,  
- with gold chunks chosen as the specific pieces of evidence you’d want the retriever to find for that question.

If you tell me one patient ID and one concrete question you care about, I can walk you through how to find and label gold_chunk_ids for that specific case.

Let's try '41681ed6-efc5-94c0-1bc0-f60b34dbd31b':

Look at data/interim/sample50/41681ed6-efc5-94c0-1bc0-f60b34dbd31b

For this patient, “Summarize the patient’s oncology history” is actually a useful **negative** test case: if there truly is no cancer-related information, your gold set for that question should be **empty**, and a good retrieval + RAG pipeline should retrieve nothing clearly oncology-related and answer “I don’t know” or “No oncology history documented.” [pmc.ncbi.nlm.nih](https://pmc.ncbi.nlm.nih.gov/articles/PMC4457181/)

Here’s how to handle it.

***

## 1. Confirm there is no oncology content

Since your ingestion and chunking pipeline already tags oncology text (using `ONCOLOGY_TERMS` and `is_oncology` flags), you can double-check from `metadata.db` that this patient really has no oncology chunks:

In a notebook: ...

If `len(onc_chunks)` is 0 and your visual inspection of `data/interim/sample50/...` matches that, you can treat this patient as “no oncology history documented” for the purpose of this test. 

***

## 2. Define ground truth for this negative question

For the question:

> “Summarize the patient’s oncology history.”

your ground-truth row would look like:

```python
{
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": [],  # empty list on purpose
}
```

Interpretation:

- There are **no chunks** that contain oncology history for this patient.
- That’s valid: some questions have no relevant evidence in the corpus. [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

When you compute Hit@K and MRR:

- Hit@K for this question should be 0 if retrieval correctly avoids oncology-marked chunks.
- MRR@K will be 0 because there is no relevant chunk to find.

This kind of negative case is useful to:

- check that your `is_oncology` filters and lexical/semantic search don’t invent cancer history where none exists;
- evaluate whether the LLM respects the “If the answer is not found in the context, respond with 'I don't know.'” instruction. [medium](https://medium.com/@akankshasinha247/rag-series-part-5-evaluating-rag-pipelines-for-groundedness-relevance-and-factuality-d67031b18d0b)

***

## 3. How many questions like this?

For your overall ground truth:

- include **a few negative questions** like this (no gold chunks), for patients without oncology or without a particular attribute;
- most questions should still have 1–3 gold chunks (positive cases), so your Hit@K and MRR measure “did we find the right evidence when it exists?” [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

A good mix for your 9 patients might be:

- 2–3 oncology questions per patient who actually has cancer-related data (positive questions),
- 1–2 “no oncology history” or “no X documented” questions for patients where that’s true (negative questions).

***

So for this specific patient and question:

- it’s a **negative ground-truth row** with `gold_chunk_ids` = `[]`,
- and it will help you evaluate whether retrieval and generation correctly say “there is no oncology history documented” rather than hallucinate one.

In [21]:
# double-check from metadata.db that this patient really has no oncology chunks:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_id = "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

conn = sqlite3.connect(db_path)

chunks_df = pd.read_sql_query(
    """
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id = ?
    """,
    conn,
    params=[patient_id],
)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,973694e5061b9103d1b1da41a6fb44459e220b91,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,55b7724042878660090ad81894f826b359e296fe,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,258c58b10c5ee39c96b9b4f2d09735a50312bd14,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
3,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,8054ebc26affb0a53d95d7436d795a985dc4ef9b,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,b60783b603a902e86bac07eaef38c50509a72cc0,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...


Number of rows: 1641


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
6,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,d0c6336772cc8ee0cc79cc2628bdba3218017dc8,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
52,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,d1ce3fd72756c7deeb385af432c9d7f00ae6bfef,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1084,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,observations,observations.csv,observations,cea4ce538b38f22f5530ac5dee22b00ba4cec3a3,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1085,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,observations,observations.csv,observations,00ec62530719197e41f04fb4c867bc31ca526aed,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1086,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,observations,observations.csv,observations,0093b36e5a57e2ba51da175385855bb663e02a7e,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...


Number of oncology chunks: 116


In [ ]:
# For this patient
onc_chunks_patient = onc_chunks  # already filtered by patient_id

# Look at a handful first to see the pattern
display(onc_chunks_patient[["doc_type", "title", "heading", "chunk_id", "chunk_text"]].head(20))

,doc_type,title,heading,chunk_id,chunk_text
6,conditions,conditions.csv,conditions,d0c6336772cc8ee0cc79cc2628bdba3218017dc8,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
52,conditions,conditions.csv,conditions,d1ce3fd72756c7deeb385af432c9d7f00ae6bfef,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1084,observations,observations.csv,observations,cea4ce538b38f22f5530ac5dee22b00ba4cec3a3,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1085,observations,observations.csv,observations,00ec62530719197e41f04fb4c867bc31ca526aed,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1086,observations,observations.csv,observations,0093b36e5a57e2ba51da175385855bb663e02a7e,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1087,observations,observations.csv,observations,3c9737d32d2ee8aa6c6b829261d331c1f1af03e1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1088,observations,observations.csv,observations,d54578d00aadb8922d938b2f9f54f8abc5d4a6be,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1090,observations,observations.csv,observations,9739bb9966595b8eaf459acd31f4d82fb2f13abc,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1094,observations,observations.csv,observations,da690e20c4e69d15ff15f5b46eb33a9f6cb33e1d,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1095,observations,observations.csv,observations,9161513ac46e5ed5a974e84cc79620f30cfffb40,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...


In [ ]:
onc_chunks_patient

That’s actually great: it means this patient is a **positive oncology case**, so “Summarize the patient’s oncology history” is a very suitable evaluation question, and you’ll be able to define meaningful `gold_chunk_ids` for it.

Here’s how to proceed.

***

## 1. Use those 116 oncology chunks to pick gold ones

Right now, `onc_chunks` holds all chunks for this patient with `is_oncology == 1`. Among those 116, you want to identify the **core evidence** for an oncology history summary:

- chunks that describe:
  - cancer diagnoses (types, sites, staging),
  - key events (diagnosis date, recurrences, metastases),
  - treatments (surgery, chemo, radiation, targeted therapy),
  - perhaps important diagnostic reports and timelines. [pmc.ncbi.nlm.nih](https://pmc.ncbi.nlm.nih.gov/articles/PMC4457181/)

In your notebook, you can inspect them like:

```python
# For this patient
onc_chunks_patient = onc_chunks  # already filtered by patient_id

# Look at a handful first to see the pattern
display(onc_chunks_patient[["doc_type", "title", "heading", "chunk_id", "chunk_text"]].head(20))
```

Then:

- Scan for chunks in `doc_type` like `oncology_timeline`, `oncology_timeline_events`, `conditions`, `diagnostic_reports`, `procedures` — those are likely to contain the most useful oncology history.  
- Within those, read `chunk_text` to find chunks that:
  - talk about diagnosis and staging,
  - list major treatments and dates,
  - summarize progression.

Pick **a small set** (say 2–5) that together give you enough information to answer:

> “Summarize the patient’s oncology history.”

Those chunk IDs become your `gold_chunk_ids`.

***

## 2. Add this to your ground truth

For example, suppose you identify:

```python
gold_ids_oncology_history = [
    "chunk-id-diagnosis",
    "chunk-id-timeline",
    "chunk-id-treatment",
]
```

Then your row would be:

```python
{
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": gold_ids_oncology_history,
}
```

Now this question:

- is a **positive** case with multiple relevant chunks,
- and your Hit@K / MRR will measure whether retrieval brings at least one of these chunks near the top. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

***

## 3. Why this is useful for tuning

Questions like this are particularly valuable because:

- they exercise both **lexical** (exact term matches like “cancer”, “stage II”) and **semantic** retrieval (synonyms, varied phrasing), [arxiv](https://arxiv.org/html/2603.03541v1)
- they rely heavily on `doc_type`, `is_oncology`, and date metadata — exactly the metadata you worked to preserve in your ingestion pipeline. [codersarts](https://www.codersarts.com/post/building-a-metadata-aware-ingestion-retrieval-pipeline)
- they show you how well hybrid search and filters work across complex, multi-chunk answers.

If you’d like, next step could be to pick one specific chunk from `oncology_timeline` and one from `conditions` for this patient and I can help you turn them into concrete `gold_chunk_ids` for your ground truth.

Perfect; those two chunk IDs are exactly the kind of “gold nuggets” you want for your oncology-history question.

For this patient:

- `4412a94aa2cefe467b43d2f7588722ab03167c6e` – a chunk from `oncology_timeline` that describes her cancer.  
- `d0c6336772cc8ee0cc79cc2628bdba3218017dc8` – a chunk from `conditions` describing the cancer diagnosis.  

Together, they give enough evidence to answer:

> “Summarize the patient’s oncology history.”

So you can define your ground-truth row like:

```python
gt_row_oncology_history = {
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": [
        "4412a94aa2cefe467b43d2f7588722ab03167c6e",
        "d0c6336772cc8ee0cc79cc2628bdba3218017dc8",
    ],
}
```

Then:

- Hit@K for this question is 1 if **either** of these chunk IDs appears in the top K retrieved results;  
- MRR@K uses the **rank of the first of these two** that appears in the top K. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

You can add `gt_row_oncology_history` to your `gt_rows` list and start using it to compare lexical vs semantic vs hybrid retrieval for this patient.

# Test 1 patient, 1 question, 2 chunks

In [22]:
gt_row_oncology_history = {
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": [
        "4412a94aa2cefe467b43d2f7588722ab03167c6e",
        "d0c6336772cc8ee0cc79cc2628bdba3218017dc8",
    ],
}

In [23]:
import math
import rag as rag

def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
    """Run the chosen search and return top-k chunk_ids in rank order."""
    if search_type == "lexical":
        results = rag.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = rag.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = rag.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    return [doc["chunk_id"] for doc in results]

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
    relevant = set(relevant_ids)
    top_k = retrieved_ids[:k]
    return int(any(doc_id in relevant for doc_id in top_k))

def mrr_at_k(retrieved_ids, relevant_ids, k):
    """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def eval_search_type(gt_row_oncology_history, search_type, k=5):
    hits = []
    mrrs = []

    for row in gt_row_oncology_history:
        patient_id = row["patient_id"]
        question = row["question"]
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

    return hit_rate, mrr

# Evaluate lexical, semantic, and hybrid for K=5
for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type([gt_row_oncology_history], st, k=5)
    print(f"{st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

lexical: Hit@5 = 0.000, MRR@5 = 0.000
semantic: Hit@5 = 0.000, MRR@5 = 0.000
hybrid: Hit@5 = 0.000, MRR@5 = 0.000


All zeros mean that, for this specific question and K=5, none of the top‑5 retrieved chunks (for lexical, semantic, or hybrid) contained any chunks, so Hit@5 and MRR@5 both evaluate to 0.

Getting zeros at first is normal — it tells you:
 - this query is hard for your current retrieval setup,
 - it’s a good candidate for tuning:
    - adjust boost_dict in search() (e.g., give more weight to heading or doc_type),
    - try different rrf_k in hybrid_search(),
    - or experiment with filtering on doc_type or is_oncology for this question.

    

1.  First check what your search functions are actually returning. Do the two gold chunk IDs appear at all? If they only appear beyond rank 5, Hit@5 and MRR@5 will be 0, even though retrieval works somewhat.

In [24]:
question = gt_row_oncology_history["question"]
patient_id = gt_row_oncology_history["patient_id"]
gold_ids = set(gt_row_oncology_history["gold_chunk_ids"])

for st in ["lexical", "semantic", "hybrid"]:
    results = rag.search(question, patient_id=patient_id, num_results=10) if st == "lexical" else \
              rag.semantic_search(question, patient_id=patient_id, num_results=10) if st == "semantic" else \
              rag.hybrid_search(question, patient_id=patient_id, num_results=10)

    print(f"\n{st.upper()} top 10 chunk_ids:")
    for rank, doc in enumerate(results, start=1):
        cid = doc["chunk_id"]
        mark = " <-- GOLD" if cid in gold_ids else ""
        print(f"{rank}: {cid}{mark}")


LEXICAL top 10 chunk_ids:
1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
2: 6de9ad49acc6eaf72f6f9490837fd3cf1482f5d4
3: ded7f386dd5af59ce715fa6ca0a5cd8df1359b17
4: 0b13f12069a342ed9a671de226d1378fabeed089
5: 0eb007dc650d4e5da238ea21a804db7ac6284c71
6: 2649097cd5f3df6c62f85b9fc9d8331057c57827
7: 0590211551c10b6259f5a9062003667e93e8f4e6
8: 337e5a497cfd365de578eb2d70a9a8e5ce1b1be8
9: bd3f64eab5732a617f775caca585388e20b7ba2c
10: a7af7e0af688eaa2d25ae618de85e72d7be0e98c

SEMANTIC top 10 chunk_ids:
1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
2: 9317205de02c6aea7e12dc5d3b66c1f1542acfb1
3: 8e56262bd19ec431b869e7f7c27325ff99ecb3bf
4: 2649097cd5f3df6c62f85b9fc9d8331057c57827
5: 55253b39fc3a7cd3bbd91cd08ec1c21d076d5fc8
6: 9f458a6e73124021d5f0a899087e67059ddae149
7: 4fcfe36428998e8c919d8f5caca637716db257f0
8: 791b3dbd2f8a11a10fa2fbdf07869dc49f62c068
9: 250dc0cc0885b6032e6801bc4ef4bc8bcf45c81a
10: a4cfc966af3919383791ed67e603839a7d41411f

HYBRID top 10 chunk_ids:
1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38

2. Try evaluating at a larger K (e.g., 10 or 20).

If Hit@10 > 0 and MRR@10 > 0, that tells you:
- the retriever can find the oncology chunks, but not in the top 5;
- you may need to:
    - increase K during retrieval,
    - or tune lexical/semantic parameters so those chunks rise in rank

In [25]:
for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type([gt_row_oncology_history], st, k=20)
    print(f"{st}: Hit@20 = {hr:.3f}, MRR@20 = {mrr:.3f}")

lexical: Hit@20 = 1.000, MRR@20 = 0.077
semantic: Hit@20 = 0.000, MRR@20 = 0.000
hybrid: Hit@20 = 0.000, MRR@20 = 0.000


So now we see that for this question, lexical search eventually finds a gold chunk but only after rank 13, so Hit@5/MRR@5 are 0, and Hit@20=1, MRR@20≈1/13≈0.077.

Semantic and hybrid search never retrieve either gold chunk in the top 20 for this query, so their Hit@K and MRR@K stay 0.

That means the evaluation code is working, and your retrieval pipeline is behaving like this:
- lexical: weak but not completely failing,
- semantic: not finding oncology-history chunks for this query,
- hybrid: dominated by whatever lexical/semantic return, so it also misses the gold chunks.

## If Hit@20 and MRR@20 were still zeros:

3. Confirm the gold IDs are correct for this index

They must be present in vector_index_metadata.json and in the minsearch_documents.json / metadata.db that rag.py loads for this environment.


In [26]:
import json
from pathlib import Path

vector_meta = json.loads(Path("../data/retrieval/vector_index_metadata.json").read_text())
docs = vector_meta["documents"]
ids_in_vector = {doc["chunk_id"] for doc in docs}

for cid in gold_ids:
    print(cid, cid in ids_in_vector)

4412a94aa2cefe467b43d2f7588722ab03167c6e True
d0c6336772cc8ee0cc79cc2628bdba3218017dc8 True


4. Confirm that search is returning anything at all


In [27]:
question = gt_row_oncology_history["question"]
patient_id = gt_row_oncology_history["patient_id"]

for st in ["lexical", "semantic", "hybrid"]:
    if st == "lexical":
        results = rag.search(
            query=question,
            patient_id=patient_id,
            num_results=10,
        )
    elif st == "semantic":
        results = rag.semantic_search(
            query=question,
            patient_id=patient_id,
            num_results=10,
        )
    else:
        results = rag.hybrid_search(
            query=question,
            patient_id=patient_id,
            num_results=10,
        )

    print(f"\n{st.upper()} — number of results:", len(results))
    for i, doc in enumerate(results, start=1):
        print(
            f"{i}: chunk_id={doc.get('chunk_id')}, "
            f"patient_id={doc.get('patient_id')}, "
            f"doc_type={doc.get('doc_type')}, "
            f"title={doc.get('title')}, "
            f"is_oncology={doc.get('is_oncology')}"
        )


LEXICAL — number of results: 10
1: chunk_id=e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=oncology_timeline, title=Oncology Timeline: Beth967 Cremin516, is_oncology=1
2: chunk_id=6de9ad49acc6eaf72f6f9490837fd3cf1482f5d4, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=patient_overview, title=Patient Overview: Beth967 Cremin516, is_oncology=1
3: chunk_id=ded7f386dd5af59ce715fa6ca0a5cd8df1359b17, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=patient_overview, title=Patient Overview: Beth967 Cremin516, is_oncology=1
4: chunk_id=0b13f12069a342ed9a671de226d1378fabeed089, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=oncology_timeline, title=Oncology Timeline: Beth967 Cremin516, is_oncology=1
5: chunk_id=0eb007dc650d4e5da238ea21a804db7ac6284c71, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=patient_overview, title=Patient Overview: Beth967 Cremin516, is_oncology=1
6: chunk_id=2649097cd5f3df6c

5. Check if the indices were built from a different manifest. It may be that:
- the current metadata.db / vector_index_metadata.json do not contain any chunks for this patient_id; or
- they contain chunks, but your rag module in the notebook is loading an older index file.

In [28]:
# Check lexical documents (if you have minsearch_documents.json)
from pathlib import Path
import json

docs_path = Path("../data/retrieval/minsearch_documents.json")
if docs_path.exists():
    min_docs = json.loads(docs_path.read_text())
    patient_ids_lexical = {doc["patient_id"] for doc in min_docs}
    print("Patient in lexical docs:", patient_id in patient_ids_lexical)

# Check semantic documents (vector index metadata)
vec_meta_path = Path("../data/retrieval/vector_index_metadata.json")
vec_meta = json.loads(vec_meta_path.read_text())
vec_docs = vec_meta["documents"]
patient_ids_vector = {doc["patient_id"] for doc in vec_docs}
print("Patient in vector docs:", patient_id in patient_ids_vector)

Patient in lexical docs: True
Patient in vector docs: True


6. Verify that rag in your notebook is loading the right files

In [29]:
import rag
from pathlib import Path

print("Lexical index path:", rag.INDEX_PATH, "exists:", Path(rag.INDEX_PATH).exists())
print("Vector index path:", rag.VECTOR_INDEX_PATH, "exists:", Path(rag.VECTOR_INDEX_PATH).exists())
print("Vector metadata path:", rag.VECTOR_METADATA_PATH, "exists:", Path(rag.VECTOR_METADATA_PATH).exists())

Lexical index path: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval/minsearch_index.pkl exists: True
Vector index path: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval/vector_index.npz exists: True
Vector metadata path: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval/vector_index_metadata.json exists: True


If all three search modes return zero results for this patient and question, even though the patient’s chunks are in the indices, then some filter condition is excluding every document. In your code, the only filters are on patient_id, doc_type, and is_oncology, so either:
- patient_id in the docs does not exactly match the string you’re passing, or
- for lexical search, minsearch’s filter_dict filtering is stricter than expected, causing it to drop everything.

In [30]:
import rag

# Check lexical docs (minsearch documents)
lex_docs = rag.index.docs  # or rag.index.docs depending on minsearch version
print("Total lexical docs:", len(lex_docs))

# Filter docs by patient id as stored in the index
lex_patient_docs = [d for d in lex_docs if d.get("patient_id") == "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"]
print("Lexical docs for this patient:", len(lex_patient_docs))
print(lex_patient_docs[:3])

Total lexical docs: 117021
Lexical docs for this patient: 1641
[{'id': '0458f43f3038f41f95741728d84d2afb01297836', 'chunk_id': '0458f43f3038f41f95741728d84d2afb01297836', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163e0abeac8d66324a6969f55b32afd7028c8', 'doc_type': 'oncology_timeline_events', 'title': 'oncology_timeline_events.csv', 'heading': 'oncology_timeline_events', 'chunk_text': 'event_type: Condition; date: 1982-04-08T21:48:26-05:00; label: Acute myeloid leukemia, disease (disorder); status: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json', 'chunk_index': 0, 'is_oncology': '1', 'date_start': '1982-04-08T21:48:26-05:00', 'date_end': '1982-04-08T21:48:26-05:00'}, {'id': 'bcc330d0e3e803819caec7ebc2cc3e5f6f4f6d9c', 'chunk_id': 'bcc330d0e3e803819caec7ebc2cc3e5f6f4f6d9c', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163

If len(lex_patient_docs) is 0 here (even though Patient in lexical docs: True from the JSON file check), it means the patient_id values actually stored in the minsearch index differ from the string you’re using (e.g., whitespace, different casing, or a different ID altogether).

Do a quick “what patient IDs exist?”:

In [31]:
patient_ids_in_index = {d.get("patient_id") for d in lex_docs}
print("Some patient_ids in index:", list(patient_ids_in_index)[:10])
# Compare this to "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

Some patient_ids in index: ['ee23ebc7-cc3c-862c-03bc-46e9321c0df1', '6e424da5-2702-49de-4046-868580fe7235', '25197dc8-9425-1999-5914-f2171b0d4e32', '8629e803-6f0f-d985-e314-ff0808ac8bd4', '43a372f7-4803-75a8-9f34-45f6d377589f', 'deea7c91-1fca-a3a2-6c68-c7acced25766', '6fb374e8-33aa-a5ea-f050-b61394dfcb99', 'e595cd98-c5d0-2a9d-ea2e-a1dc325005fa', '64ae3769-65e4-222e-6793-1a3bc14ec682', '397b2de6-ccd8-858f-bf4a-b6fc379589bd']


In [32]:
vec_docs = rag.vector_documents
vec_patient_docs = [d for d in vec_docs if d.get("patient_id") == "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"]
print("Vector docs for this patient:", len(vec_patient_docs))
print(vec_patient_docs[:3])

Vector docs for this patient: 1641
[{'chunk_id': '0458f43f3038f41f95741728d84d2afb01297836', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163e0abeac8d66324a6969f55b32afd7028c8', 'doc_type': 'oncology_timeline_events', 'title': 'oncology_timeline_events.csv', 'heading': 'oncology_timeline_events', 'chunk_text': 'event_type: Condition; date: 1982-04-08T21:48:26-05:00; label: Acute myeloid leukemia, disease (disorder); status: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json', 'chunk_index': 0, 'is_oncology': 1, 'date_start': '1982-04-08T21:48:26-05:00', 'date_end': '1982-04-08T21:48:26-05:00'}, {'chunk_id': 'bcc330d0e3e803819caec7ebc2cc3e5f6f4f6d9c', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163e0abeac8d66324a6969f55b32afd7028c8', 'doc_type': 'oncology_timeline_events', 'title': 'oncology_timeline_events.csv', 'heading': '

In [33]:
patient_ids_in_index = {d.get("patient_id") for d in vec_docs}
print("Some patient_ids in index:", list(patient_ids_in_index)[:10])
# Compare this to "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

Some patient_ids in index: ['ee23ebc7-cc3c-862c-03bc-46e9321c0df1', '6e424da5-2702-49de-4046-868580fe7235', '25197dc8-9425-1999-5914-f2171b0d4e32', '8629e803-6f0f-d985-e314-ff0808ac8bd4', '43a372f7-4803-75a8-9f34-45f6d377589f', 'deea7c91-1fca-a3a2-6c68-c7acced25766', '6fb374e8-33aa-a5ea-f050-b61394dfcb99', 'e595cd98-c5d0-2a9d-ea2e-a1dc325005fa', '64ae3769-65e4-222e-6793-1a3bc14ec682', '397b2de6-ccd8-858f-bf4a-b6fc379589bd']


2. Test search without the patient filter to see whether query + index works at all.

If you get non-zero results here, then:
- your retrieval logic works,
- and the patient filter is the thing eliminating everything.


In [34]:
question = gt_row_oncology_history["question"]

# Lexical without patient filter
lex_results_no_filter = rag.search(
    query=question,
    patient_id=None,   # Force no filter
    num_results=10,
)
print("LEXICAL (no patient filter), n:", len(lex_results_no_filter))
print([r["patient_id"] for r in lex_results_no_filter])

# Semantic without patient filter
sem_results_no_filter = rag.semantic_search(
    query=question,
    patient_id=None,
    num_results=10,
)
print("SEMANTIC (no patient filter), n:", len(sem_results_no_filter))
print([r["patient_id"] for r in sem_results_no_filter])

LEXICAL (no patient filter), n: 10
['66e681dd-d945-9aba-95cb-d9606594cc9c', 'db62bf97-9977-e94c-94d1-46fa7abc524f', 'f18bd61f-f7be-ef83-db73-bbe83fb3391c', 'd44ac721-88ff-81cc-dd2f-7a30a1c1a221', '66d5d38f-4e52-3896-822a-541e29cd965b', 'cf018f86-f6b8-12a1-9f4d-ba14ff081fb4', '940b81eb-27fb-52d3-251e-df6edb18d893', '3af995f1-02a5-07ee-5a7e-e2470a017f1e', '8629e803-6f0f-d985-e314-ff0808ac8bd4', '8dfff5d0-a8f1-4eac-3987-1670cc41239b']
SEMANTIC (no patient filter), n: 10
['b73cbc40-ef87-47d5-08dc-4288048dbf1c', 'b31cb2e6-f4db-0f20-43e7-6ab691c2a32b', '0c0f2095-e8ab-7ac4-6ef4-625748255480', 'e595cd98-c5d0-2a9d-ea2e-a1dc325005fa', 'b9b9379e-68a1-49a5-4292-5cc3380e30d2', '64ae3769-65e4-222e-6793-1a3bc14ec682', 'b9b9379e-68a1-49a5-4292-5cc3380e30d2', '8dfff5d0-a8f1-4eac-3987-1670cc41239b', 'c457bf0f-0a34-2ebf-cec6-a385fb33d8a8', '0f5704ee-b38b-5a68-449d-9c44806517d0']


# END

After the pipeline works, run one final pass on the full 258-patient corpus to report scalability or at least show that the same code works end-to-end.[confluence.hl7]

A useful pattern is to keep:
patients_50/ for development,
patients_258/ for final scalability testing,
and a manifest file documenting the chosen IDs and selection criteria.
